# Describe and Compare Sales Orders

Run `src/prepare_sales.py` first to create `data/sales_clean.csv`. Run cells in order. The notebook creates its own variables independently of the script Console. Stop before the Plotly section until you add that dependency.

## Load the Prepared Data

Use the project environment. The path check supports running from the project root or this notebooks folder.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd()
if not (project_root / "data").is_dir():
    project_root = project_root.parent
sales = pd.read_csv(
    project_root / "data/sales_clean.csv", parse_dates=["order_date"]
)

## Describe the Data

Compare mean and median order values, and check how many retail and wholesale orders you have.

In [ ]:
print(sales[["quantity", "unit_price", "sale_price"]].describe().round(2))
print(sales["order_type"].value_counts())

## Compare Typical Orders

Calculate the median order value within each product category and order type. Keep group counts and quantity and unit-price medians for context.

In [ ]:
order_summary = sales.groupby(
    ["product_category", "order_type"], as_index=False
).agg(
    orders=("order_number", "size"),
    median_order_value=("sale_price", "median"),
    median_quantity=("quantity", "median"),
    median_unit_price=("unit_price", "median"),
)
print(order_summary.round(2).to_string(index=False))

## Visualize the Comparison

Compare the retail and wholesale bars within each category. These figures describe a fictional dataset.

In [ ]:
chart_data = order_summary.pivot(
    index="product_category", columns="order_type", values="median_order_value"
)
ax = chart_data.plot.bar(rot=0, figsize=(8, 4))
ax.set(xlabel="Product category", ylabel="Median order value")
ax.set_title("Retail and Wholesale Order Values")
plt.tight_layout()
plt.show()

## Add an Interactive Chart

Before running the next cell, run `uv add plotly==7.0.0` in the project terminal. If you restart the notebook kernel afterward, rerun the preceding cells. Hover over each bar to see the group count and median quantity.

In [ ]:
import plotly.express as px

chart = px.bar(
    order_summary,
    x="product_category",
    y="median_order_value",
    color="order_type",
    barmode="group",
    hover_data=["orders", "median_quantity"],
    labels={"median_order_value": "Median order value"},
    title="Retail and Wholesale Order Values",
)
chart.show()

## Optional Assistant Follow-Up

Ask whether larger quantities help explain the retail/wholesale differences. Check unit prices and product mix as well. Group medians alone do not establish a cause, and multiplying medians does not generally give the median of a product. The starting calculation to verify is already in `order_summary`.